<a href="https://colab.research.google.com/github/haticeeeayd/customer-journey-touchpoint-analytics/blob/main/customerjourney.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/datasets/Customer Journey/"

train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")

df = pd.concat([train, test], ignore_index=True)

print(f"Train: {len(train):,} rows")
print(f"Test:  {len(test):,} rows")
print(f"Total: {len(df):,} rows")

Train: 103,904 rows
Test:  25,976 rows
Total: 129,880 rows


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         129880 non-null  int64  
 1   id                                 129880 non-null  int64  
 2   Gender                             129880 non-null  object 
 3   Customer Type                      129880 non-null  object 
 4   Age                                129880 non-null  int64  
 5   Type of Travel                     129880 non-null  object 
 6   Class                              129880 non-null  object 
 7   Flight Distance                    129880 non-null  int64  
 8   Inflight wifi service              129880 non-null  int64  
 9   Departure/Arrival time convenient  129880 non-null  int64  
 10  Ease of Online booking             129880 non-null  int64  
 11  Gate location                      1298

In [16]:
df.isnull().sum()[df.isnull().sum() > 0]

,0
Arrival Delay in Minutes,393


In [17]:
# Drop unnecessary index column
df.drop(columns=["Unnamed: 0"], inplace=True)

# Fill missing Arrival Delay with median
df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(
    df["Arrival Delay in Minutes"].median()
)

print("Missing values:", df.isnull().sum().sum())
print("Total rows:", len(df))

Missing values: 0
Total rows: 129880


In [18]:
def age_group(age):
    if age <= 25:
        return "18-25 Young"
    elif age <= 35:
        return "26-35 Young Adult"
    elif age <= 50:
        return "36-50 Middle Age"
    else:
        return "51+ Senior"

df["Age_Group"] = df["Age"].apply(age_group)

df["Age_Group"].value_counts()

,count
Age_Group,
36-50 Middle Age,44533
51+ Senior,33602
18-25 Young,28173
26-35 Young Adult,23572


In [19]:
touchpoints = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

df["Avg_Touchpoint_Score"] = df[touchpoints].mean(axis=1)
df["CSAT_Score"] = ((df["Avg_Touchpoint_Score"] - 1) / 4 * 100).round(1)

print(df[["Avg_Touchpoint_Score", "CSAT_Score"]].describe().round(2))

       Avg_Touchpoint_Score  CSAT_Score
count             129880.00   129880.00
mean                   3.24       56.03
std                    0.66       16.54
min                    1.07        1.80
25%                    2.79       44.60
50%                    3.29       57.10
75%                    3.71       67.90
max                    5.00      100.00


In [20]:
journey_stages = {
    "Stage_1_Booking": ["Ease of Online booking"],
    "Stage_2_Airport": ["Checkin service", "Gate location", "Baggage handling"],
    "Stage_3_Boarding": ["Online boarding", "Departure/Arrival time convenient"],
    "Stage_4_Inflight": ["Seat comfort", "Food and drink", "Inflight wifi service",
                          "Inflight entertainment", "On-board service",
                          "Leg room service", "Inflight service", "Cleanliness"]
}

for stage, cols in journey_stages.items():
    df[stage] = df[cols].mean(axis=1).round(2)

print("=== Customer Journey Stage Scores ===\n")
for stage in journey_stages.keys():
    print(f"{stage}: {df[stage].mean():.2f} / 5.00")

=== Customer Journey Stage Scores ===

Stage_1_Booking: 2.76 / 5.00
Stage_2_Airport: 3.31 / 5.00
Stage_3_Boarding: 3.16 / 5.00
Stage_4_Inflight: 3.30 / 5.00


In [21]:
print("=== Satisfaction Rate by Segment ===\n")

segments = ["Customer Type", "Type of Travel", "Class", "Age_Group"]

for seg in segments:
    table = df.groupby(seg)["satisfaction"].apply(
        lambda x: (x == "satisfied").mean() * 100
    ).round(1)
    print(f"--- {seg} ---")
    print(table.to_string())
    print()

=== Satisfaction Rate by Segment ===

--- Customer Type ---
Customer Type
Loyal Customer       47.8
disloyal Customer    24.0

--- Type of Travel ---
Type of Travel
Business travel    58.4
Personal Travel    10.1

--- Class ---
Class
Business    69.4
Eco         18.8
Eco Plus    24.6

--- Age_Group ---
Age_Group
18-25 Young          28.4
26-35 Young Adult    37.6
36-50 Middle Age     53.8
51+ Senior           46.4



In [11]:
# Touchpoint bazlı ortalamalar: satisfied vs dissatisfied
print("=== Touchpoint Skorları: Satisfied vs Dissatisfied ===\n")

comparison = df.groupby("satisfaction")[touchpoints].mean().T.round(2)
comparison.columns = ["Dissatisfied", "Satisfied"]
comparison["Gap"] = (comparison["Satisfied"] - comparison["Dissatisfied"]).round(2)
comparison = comparison.sort_values("Gap", ascending=False)

print(comparison.to_string())

=== Touchpoint Skorları: Satisfied vs Dissatisfied ===

                                   Dissatisfied  Satisfied   Gap
Online boarding                            2.66       4.03  1.37
Inflight entertainment                     2.89       3.96  1.07
Seat comfort                               3.04       3.97  0.93
On-board service                           3.02       3.86  0.84
Leg room service                           2.99       3.82  0.83
Cleanliness                                2.93       3.75  0.82
Inflight wifi service                      2.40       3.16  0.76
Checkin service                            3.04       3.65  0.61
Baggage handling                           3.37       3.97  0.60
Inflight service                           3.39       3.97  0.58
Food and drink                             2.96       3.53  0.57
Ease of Online booking                     2.55       3.03  0.48
Gate location                              2.98       2.97 -0.01
Departure/Arrival time convenient 

In [22]:
df["Total_Delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]

df["Satisfied"] = (df["satisfaction"] == "satisfied").astype(int)

df["Delay_Group"] = pd.cut(
    df["Total_Delay"],
    bins=[0, 1, 30, 60, 120, float("inf")],
    labels=["No Delay", "1-30 min", "31-60 min", "61-120 min", "120+ min"],
    include_lowest=True
)

print("=== Delay vs Satisfaction ===\n")
print(df.groupby("Delay_Group")["Satisfied"].mean().apply(lambda x: f"{x*100:.1f}%"))

=== Delay vs Satisfaction ===

Delay_Group
No Delay      47.1%
1-30 min      42.8%
31-60 min     36.2%
61-120 min    36.2%
120+ min      35.8%
Name: Satisfied, dtype: object


/tmp/ipykernel_567/1901887964.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("Delay_Group")["Satisfied"].mean().apply(lambda x: f"{x*100:.1f}%"))


In [24]:
output_path = "/content/drive/MyDrive/datasets/Customer Journey/airline_clean.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved: {len(df):,} rows, {len(df.columns)} columns")
print(f"Path: {output_path}")

Saved: 129,880 rows, 34 columns
Path: /content/drive/MyDrive/datasets/Customer Journey/airline_clean.csv


In [26]:
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.replace("/", "_")
df.columns = df.columns.str.replace("-", "_")

output_path = "/content/drive/MyDrive/datasets/Customer Journey/airline_clean.csv"
df.to_csv(output_path, index=False)
print("Saved!")

Saved!
